# Attack Label Analysis

This notebook analyses the enriched labeled files across all four test datasets:
- `hai-test1-labeled.csv` — HAI physical layer, test split 1
- `hai-test2-labeled.csv` — HAI physical layer, test split 2
- `end-test1-labeled.csv` — HAIEnd DCS layer, test split 1
- `end-test2-labeled.csv` — HAIEnd DCS layer, test split 2

**Questions answered:**
1. What fraction of each file is under attack vs normal?
2. How do attacks distribute over time in each file?
3. What is the breakdown of AP vs AE attacks in test1 vs test2?
4. Which attacks appear in HAI only vs both HAI and HAIEnd, and why?

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.dates as mdates
import seaborn as sns

sns.set_theme(style='whitegrid', font_scale=1.1)

LABELED_DIR = os.environ.get('OUT_DIR', 'C:/Users/farah/OneDrive/Desktop/AI_project/labeled')

files = {
    'hai-test1': (os.path.join(LABELED_DIR, 'hai-test1-labeled.csv'), 'timestamp'),
    'hai-test2': (os.path.join(LABELED_DIR, 'hai-test2-labeled.csv'), 'timestamp'),
    'end-test1': (os.path.join(LABELED_DIR, 'end-test1-labeled.csv'), 'Timestamp'),
    'end-test2': (os.path.join(LABELED_DIR, 'end-test2-labeled.csv'), 'Timestamp'),
}

dfs = {}
for name, (path, ts_col) in files.items():
    df = pd.read_csv(path, low_memory=False, parse_dates=[ts_col])
    df = df.rename(columns={ts_col: 'timestamp'})
    dfs[name] = df
    print(f'{name}: {len(df):,} rows, columns={df.shape[1]}')

## 1. Overview — Attack vs Normal row counts

In [ ]:
rows = []
for name, df in dfs.items():
    total = len(df)
    attack = (df['attack_type'] != 'normal').sum()
    normal = total - attack
    rows.append({
        'file': name,
        'total': total,
        'normal': normal,
        'attack': attack,
        'attack_%': round(100 * attack / total, 2)
    })

overview = pd.DataFrame(rows)
print(overview.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

x = np.arange(4)
w = 0.35
ax.bar(x - w/2, overview['normal'],  w, label='Normal', color='steelblue')
ax.bar(x + w/2, overview['attack'],  w, label='Attack',  color='tomato')

for i, row in overview.iterrows():
    ax.text(i + w/2, row['attack'] + 500, f"{row['attack_%']}%",
            ha='center', va='bottom', fontsize=9, color='tomato')

ax.set_xticks(x)
ax.set_xticklabels(overview['file'])
ax.set_ylabel('Rows (1-second samples)')
ax.set_title('Normal vs Attack rows per file')
ax.legend()
plt.tight_layout()
plt.show()

**Observation:** Attacks make up only a small fraction of each file (~2-4%). 
Test2 has far more rows (230,400 vs 54,000) because more attack scenarios were included and more normal operation time was recorded.

## 2. Attack Timeline — When do attacks occur in each file?

In [ ]:
TYPE_COLOR = {'normal': 'lightgrey', 'AP': 'steelblue', 'AE': 'tomato'}

fig, axes = plt.subplots(4, 1, figsize=(14, 8), sharex=False)

for ax, (name, df) in zip(axes, dfs.items()):
    colors = df['attack_type'].map(TYPE_COLOR)
    ax.scatter(df['timestamp'], np.zeros(len(df)),
               c=colors, s=0.3, alpha=0.6)
    ax.set_ylabel(name, fontsize=9)
    ax.set_yticks([])
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d %H:%M'))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=20, ha='right', fontsize=7)

patches = [mpatches.Patch(color=c, label=l) for l, c in TYPE_COLOR.items()]
fig.legend(handles=patches, loc='upper right', fontsize=9)
fig.suptitle('Attack timeline — each dot = 1 second', fontsize=12)
plt.tight_layout()
plt.show()

**Observation:** 
- **test1** is a single overnight session (Aug 12–13): all attacks are AP (blue), evenly spaced.
- **test2** spans several days (Aug 17–19): AP attacks dominate, with 8 AE attacks (red) clustered on Aug 18–19.
- **HAI and HAIEnd timelines are identical** for the same split — every attack window appears in both files.

## 3. Test1 vs Test2 — Attack type breakdown

In [ ]:
# Count unique attack instances (not rows) per split
for split_name, name in [('Test 1', 'hai-test1'), ('Test 2', 'hai-test2')]:
    df = dfs[name]
    attacks = df[df['attack_type'] != 'normal'].drop_duplicates('attack_id')
    print(f"\n=== {split_name} ({name}) ===")
    summary = attacks.groupby(['attack_type', 'combination']).size().reset_index(name='count')
    summary['label'] = summary['attack_type'] + ' / combination=' + summary['combination']
    print(summary[['label', 'count']].to_string(index=False))
    print(f"Total unique attacks: {len(attacks)}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

split_labels  = ['Test 1\n(A101-A114)', 'Test 2\n(A201-A238)']
ap_primitive  = [14, 30]   # AP, combination=no
ap_combo      = [0,  10]   # AP, combination=yes
ae_attacks    = [0,  8]    # AE

x = np.arange(2)
w = 0.25

axes[0].bar(x - w, ap_primitive, w, label='AP primitive', color='steelblue')
axes[0].bar(x,     ap_combo,     w, label='AP combination', color='orange')
axes[0].bar(x + w, ae_attacks,   w, label='AE',            color='tomato')
axes[0].set_xticks(x)
axes[0].set_xticklabels(split_labels)
axes[0].set_ylabel('Number of attack instances')
axes[0].set_title('Attack instances by type')
axes[0].legend(fontsize=8)

# Pie chart for test2 composition
axes[1].pie([30, 10, 8],
            labels=['AP primitive', 'AP combination', 'AE'],
            colors=['steelblue', 'orange', 'tomato'],
            autopct='%1.0f%%', startangle=90)
axes[1].set_title('Test 2 attack composition (38 instances)')

plt.suptitle('Test1 = 14 AP attacks only   |   Test2 = 38 attacks (AP + AE + combinations)', fontsize=10)
plt.tight_layout()
plt.show()

**Key difference between Test1 and Test2:**

| | Test 1 | Test 2 |
|---|---|---|
| AP primitive | 14 | 30 |
| AP combination | 0 | 10 |
| AE | 0 | 8 |
| **Total** | **14** | **48** |

Test 1 is a warm-up: only simple AP attacks, one controller at a time.  
Test 2 is the full challenge: adds AE attacks (internal DCS) and combination attacks (two controllers hit simultaneously).

## 4. HAI vs HAIEnd — Which attacks appear where and why?

In [ ]:
# For test2: compare attack_id presence in hai vs end
hai2 = dfs['hai-test2']
end2 = dfs['end-test2']

hai_attacks = set(hai2[hai2['attack_type'] != 'normal']['attack_id'].unique())
end_attacks = set(end2[end2['attack_type'] != 'normal']['attack_id'].unique())

print(f"Attacks in hai-test2:  {sorted(hai_attacks)}")
print(f"Attacks in end-test2:  {sorted(end_attacks)}")
print(f"\nIn HAI but not HAIEnd: {sorted(hai_attacks - end_attacks)}")
print(f"In both:               {sorted(hai_attacks & end_attacks)}")

In [ ]:
# Side-by-side timeline: HAI vs HAIEnd test2, coloured by attack_type
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 5), sharex=True)

for ax, name, label in [(ax1, 'hai-test2', 'HAI (physical I/O)'),
                         (ax2, 'end-test2', 'HAIEnd (DCS internal)')]:
    df = dfs[name]
    colors = df['attack_type'].map(TYPE_COLOR)
    ax.scatter(df['timestamp'], np.zeros(len(df)), c=colors, s=0.3, alpha=0.7)
    ax.set_ylabel(label, fontsize=9)
    ax.set_yticks([])

ax2.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d %H:%M'))
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=20, ha='right', fontsize=7)

patches = [mpatches.Patch(color=c, label=l) for l, c in TYPE_COLOR.items()]
fig.legend(handles=patches, loc='upper right', fontsize=9)
fig.suptitle('Test 2 — HAI vs HAIEnd attack timeline (red = AE attacks appear in BOTH)', fontsize=11)
plt.tight_layout()
plt.show()

**Why do AE attacks appear in both HAI and HAIEnd?**

```
AP attacks  →  target HAI I/O points directly
               HAI sees it.  HAIEnd does NOT (DCS not involved)

AE attacks  →  target HAIEnd internal DCS parameters
               HAIEnd sees it immediately.
               Effect propagates to physical sensors → HAI sees it too.
               dataset = 'both'
```

This is why `dataset` column is `hai` for AP and `both` for AE.

In [ ]:
# Show the 8 AE attacks side by side: row count in HAI vs HAIEnd
ae_ids = ['A220', 'A221', 'A222', 'A223', 'A235', 'A236', 'A237', 'A238']

rows = []
for aid in ae_ids:
    hai_rows = (dfs['hai-test2']['attack_id'] == aid).sum()
    end_rows = (dfs['end-test2']['attack_id'] == aid).sum()
    scenario = dfs['hai-test2'][dfs['hai-test2']['attack_id'] == aid]['scenario'].iloc[0]
    rows.append({'attack_id': aid, 'scenario': scenario,
                 'rows_in_HAI': hai_rows, 'rows_in_HAIEnd': end_rows})

ae_df = pd.DataFrame(rows)
print(ae_df.to_string(index=False))

## 5. Attack Duration Distribution

In [ ]:
import sys
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'hai-digital-twin'))
sys.path.insert(0, '..')

timetable = pd.read_csv('../utils/test_data.csv', parse_dates=['start', 'end'])
timetable['split_label'] = timetable['split'].map({1: 'Test 1', 2: 'Test 2'})
timetable['type_label'] = timetable.apply(
    lambda r: 'AP combination' if r['combination'] == 'yes' else r['attack_type'], axis=1
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

pal = {'AP': 'steelblue', 'AE': 'tomato', 'AP combination': 'orange'}

for ax, (split, grp) in zip(axes, timetable.groupby('split_label')):
    for ttype, sub in grp.groupby('type_label'):
        ax.hist(sub['duration_sec'], bins=15, alpha=0.7,
                color=pal.get(ttype, 'grey'), label=ttype)
    ax.set_xlabel('Duration (seconds)')
    ax.set_ylabel('Count')
    ax.set_title(f'{split} — attack durations')
    ax.legend(fontsize=8)

plt.suptitle('Most attacks are 60-200 s; AP40/AP41/AP44+AP47 are intentionally long (extended pump runs)', fontsize=9)
plt.tight_layout()
plt.show()

print(timetable.groupby('type_label')['duration_sec'].describe().round(1))

## 6. Attack density per controller

In [ ]:
# Count attack-rows per target_controller across all 4 files combined
combined = pd.concat(
    [df[['timestamp', 'attack_type', 'target_controller', 'combination']]
     for df in dfs.values()],
    ignore_index=True
)

attack_rows = combined[combined['attack_type'] != 'normal']

# For combination attacks the controller field has both ("P1-LC / P2-SC") — split and count separately
ctrl_counts = (
    attack_rows['target_controller']
    .str.split(' / ')
    .explode()
    .str.strip()
    .value_counts()
)

fig, ax = plt.subplots(figsize=(9, 4))
ctrl_counts.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_xlabel('Target controller')
ax.set_ylabel('Attack rows (1-sec samples, all 4 files)')
ax.set_title('Which controllers are targeted most across all test files?')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

## 7. Summary

| Question | Answer |
|----------|--------|
| How many rows are under attack? | ~2-4% per file — heavily imbalanced |
| Test1 vs Test2 | Test1 = 14 simple AP; Test2 = 38 attacks (AP + AE + combinations) |
| HAI vs HAIEnd | AP attacks in HAI only; AE attacks in **both** (DCS → physical propagation) |
| Attack durations | Most are 60–200 s; a few long ones (AP40/AP41/AP44+AP47) exceed 600 s |
| Most targeted controller | P1-LC and P1-PC appear most across all test files |

**For modelling:**
- Use `attack_type` as your label (`normal` / `AP` / `AE`).
- Handle class imbalance (SMOTE, class weights, or focal loss).
- AE detection requires HAIEnd features; AP detection can use HAI alone.
- Combination attacks (A224-A233) are harder — two controllers diverge simultaneously.